# 02. Credit Card Fraud Detection — Feature Engineering & Imbalanced Modeling

**Role Perspective**: Machine Learning Engineer / Data Scientist  
**Objective**: Construct leakage-safe preprocessing pipelines, compare 5 supervised models via Stratified Cross-Validation, optimize decision thresholds, and explain predictive signals.

---

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_curve
import joblib

from src.data_processing import build_preprocessor, prepare_data
from src.train import get_model_candidates, run_pipeline

## 1. Leakage-Safe Feature Engineering Pipeline
We instantiate domain transformations:
1. Cyclical trigonometric encoding for `transaction_hour` (`hour_sin`, `hour_cos`)
2. Interaction feature: `foreign_transaction * location_mismatch`
3. Log monetary scaling: `log1p(amount)`
4. Indicator flags for night window, velocity spikes, and untrusted devices
5. Exclusion of non-predictive `transaction_id`

In [ ]:
df = pd.read_csv(os.path.join('..', 'data', 'credit_card_fraud_10k.csv'))
X, y = prepare_data(df)
preprocessor = build_preprocessor()
X_trans = preprocessor.fit_transform(X)
print(f"Transformed Feature Matrix Shape: {X_trans.shape}")

## 2. Model Training & Evaluation Across 5 Algorithms
We benchmark Baseline, Logistic Regression, Decision Tree, Random Forest, and HistGradientBoosting.

In [ ]:
comp_df = pd.read_csv(os.path.join('..', 'reports', 'model_comparison.csv'))
display(comp_df[['Model', 'CV_PR_AUC', 'pr_auc', 'CV_ROC_AUC', 'roc_auc', 'recall', 'precision', 'f1_score']])

## 3. Threshold Optimization & Cost-Sensitive Analysis
In fraud detection, default 0.50 threshold is arbitrary. We evaluate performance across candidate thresholds to minimize false negative cost.

In [ ]:
model_pipe = joblib.load(os.path.join('..', 'models', 'best_fraud_model_pipeline.joblib'))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
probs = model_pipe.predict_proba(X_test)[:, 1]

from src.evaluate import evaluate_threshold_range, compute_hypothetical_cost
thresh_eval = evaluate_threshold_range(y_test.values, probs)
cost_eval = compute_hypothetical_cost(thresh_eval, cost_fp=10.0, cost_fn=200.0)
display(cost_eval.head(8))

## 4. Model Explainability & Permutation Importance
Permutation feature importance measured on the untouched test split.

In [ ]:
from sklearn.inspection import permutation_importance
perm = permutation_importance(model_pipe, X_test, y_test, scoring='average_precision', random_state=42, n_repeats=10)
sorted_idx = perm.importances_mean.argsort()[::-1]
fi_df = pd.DataFrame({
    'Feature': np.array(X.columns)[sorted_idx],
    'Mean_Importance_Drop': perm.importances_mean[sorted_idx]
}).head(10)
display(fi_df)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=fi_df, x='Mean_Importance_Drop', y='Feature', palette='mako')
plt.title('Permutation Feature Importance (Champion Model)')
plt.xlabel('Mean PR-AUC Drop')
plt.tight_layout()
plt.show()